<a href="https://colab.research.google.com/github/importYM/bigdata_analysis_engineer/blob/main/Type1_BigdataAnalysisEngineer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 빅데이터 분석기사 실기 - 제1유형 학습 노트

> 데이터 전처리 및 기술통계

## 1. 기본 환경 설정 및 데이터 로드, 출력

In [ ]:
import pandas as pd
import numpy as np

# 출력 옵션: 컬럼이 잘리지 않도록 설정
pd.set_option('display.max_columns', None)

# 데이터 불러오기
df = pd.read_csv('data.csv')

# ── 데이터 구조 탐색 ────────────────────────────────────────
print(df.shape)          # (행 수, 열 수)
print(df.columns)        # 컬럼명 목록
print(df.dtypes)         # 각 컬럼의 데이터 타입
print(df.head())         # 상위 5행 확인
print(df.tail())         # 하위 5행 확인
print(df.info())         # 결측치 수 + 타입 한번에 확인 (가장 유용)
print(df.describe())     # 수치형 컬럼의 기술통계: count(NaN 제외), mean, std, min, 25%(제1사분위수), 50%(median), 75%(제3사분위수), max
print(df.describe(include='all'))  # 숫자와 문자열 데이터를 모두 요약

# ── 반올림 및 자릿수 처리 ─────────────────────────────────────
print(round(result, 2))  # 반올림으로 소수점 2자리까지 출력
print(round(result))     # 정수로 반올림

# ── 정수 변환 출력 ──────────────────────────────────────────
print(int(result))       # 소수점 버림

## 2. 결측치 처리 (Missing Values)

In [ ]:
# ── 결측치 탐지 ─────────────────────────────────────────────
print(df.isnull().sum())        # 각 컬럼별 결측치 개수
print(df.isnull().sum().sum())  # 전체 결측치 개수

# ── 결측치 제거 (dropna) ────────────────────────────────────
df_drop_row = df.dropna()       # 결측치 있는 행 전체 삭제 (axis=0이 기본)
df_drop_col = df.dropna(axis=1) # 결측치 있는 열 전체 삭제
df_drop_sub = df.dropna(subset=['col_a'])    # 특정 컬럼에 결측치 있는 행만 삭제

# ── 결측치 대체 (fillna) ────────────────────────────────────
# (1) 수치형: 평균 / 중앙값으로 대체
df['col'] = df['col'].fillna(df['col'].mean())   # 평균으로 대체
df['col'] = df['col'].fillna(df['col'].median()) # 중앙값으로 대체

# (2) 범주형: 최빈값으로 대체
col_mode = df['city'].mode()[0] # mode()는 Series 반환→ [0]으로 첫 번째 값 선택
df['city'] = df['city'].fillna(col_mode)

# (3) 앞/뒤 값으로 채우기 (시계열 데이터에서 유용)
df['col'] = df['col'].fillna(method='ffill') # 앞 값으로 채움 (forward fill)
df['col'] = df['col'].fillna(method='bfill') # 뒤 값으로 채움 (backward fill)

## 3. 데이터 타입 확인 및 변환

In [ ]:
# ── 타입 변환 (astype) ──────────────────────────────────────
df['col'] = df['col'].astype('int')             # 단일 컬럼
df = df.astype({'col1': 'int', 'col2': 'str'})  # 여러 컬럼 일괄 변환

# ── 특정 타입 컬럼만 추출 ──────────────────────────────────────
num_cols = df.select_dtypes(include='number').columns   # int + float
obj_cols = df.select_dtypes(include='object').columns   # 문자열
cat_cols = df.select_dtypes(exclude='number').columns   # 수치형 제외 전체

# ── 고유값 탐색 ─────────────────────────────────────────────
print(df['col'].unique())         # 고유값 목록 출력
print(df['col'].nunique())        # 고유값 개수 (결측치 제외)

# value_counts: 각 값의 빈도수 출력
print(df['col'].value_counts())   # 빈도 내림차순 정렬 (기본값: ascending=False)
print(df['col'].value_counts(ascending=True))   # 빈도 오름차순 정렬
print(df['col'].value_counts(normalize=True))   # 비율로 출력 (합계=1)

## 4. 문자열 조작 및 정규표현식 (Regex)

In [ ]:
# ── 문자열 길이 계산 (str.len) ───────────────────────────────
name_len_mean = df['name'].str.len().mean()     # 이름 글자 수 평균

# ── 문자열 분리, 추출 (str.split) ────────────────────────────
# 예: 'user@gmail.com' → 'user', 'gmail.com'
df['email_id']     = df['email'].str.split('@').str[0]   # @ 앞부분
df['email_domain'] = df['email'].str.split('@').str[1]   # @ 뒷부분

# ── 문자열 변환 ─────────────────────────────────────────────
df['name'] = df['name'].str.upper()    # 대문자로
df['name'] = df['name'].str.lower()    # 소문자로
df['name'] = df['name'].str.strip()    # 앞뒤 공백 제거

In [ ]:
# ── 포함 여부 필터링 (str.contains) ──────────────────────────
#    na= False: str.contains()에서 결측치를 False로 처리 → 필터링 결과에서 제외
#    na= True : str.contains()에서 결측치를 True로 처리  → 필터링 결과에 포함
#    na 옵션 생략(기본): 결측치를 NaN로 반환 → df[...] 필터링 시 ValueError 위험

# 예: 이메일에 'gmail'이 포함된 행 개수
gmail_count = df['email'].str.contains('gmail').sum()
# regex=False: 특수문자를 정규표현식 아닌 일반 문자열로 취급
gmail_count = df['email'].str.contains('gmail.com', regex=False).sum()
# 예: 영문으로만 구성된 이름만 필터링
eng_names = df[df['name'].str.contains(r'^[A-Za-z]+$', na=False)]



```
# ── 정규표현식 패턴 정리 ──────────────────────────────────────
# str.contains(r'패턴', regex=True) 형식으로 사용 (기본값: regex=True)

# 패턴            의미
# r'^[A-Za-z]+$' 영문 알파벳으로만 구성
# r'^\d+$'       숫자로만 구성
# r'^[가-힣]+$'   한글로만 구성
# r'\d'          숫자를 하나 이상 포함
# r'\D'          숫자가 아닌 문자 포함
# r'[가-힣]'      한글을 하나라도 포함
# r'\w'          영문, 숫자, 밑줄(_) 포함
# r'\s'          공백 포함
# r'\.'          마침표(.) - 특수문자는 역슬래시 이스케이프
```



In [ ]:
# ── 문자열 치환 (str.replace) ──────────────────────────────
# 예: 하이픈 제거
df['phone'] = df['phone'].str.replace('-', '', regex=False)
# 예: 숫자를 'NUM'으로
df['text']  = df['text'].str.replace(r'\d+', 'NUM', regex=True)

## 5. 날짜 및 시간 (Datetime) 처리

In [ ]:
# ── 날짜 타입으로 변환 ───────────────────────────────────────
df['date'] = pd.to_datetime(df['date'])
df['date'] = pd.to_datetime(df['date'], format='%Y/%m/%d')  # 포맷 지정 시

# ── 날짜 성분 추출 (dt 접근자) ───────────────────────────────
df['year']     = df['date'].dt.year          # 연도 (예: 2024)
df['month']    = df['date'].dt.month         # 월 (1~12)
df['day']      = df['date'].dt.day           # 일 (1~31)
df['hour']     = df['date'].dt.hour          # 시 (0~23)
df['weekday']  = df['date'].dt.weekday       # 요일 숫자 (월=0, ..., 일=6)
df['day_name'] = df['date'].dt.day_name()    # 요일 이름 (Monday, ...)
df['quarter']  = df['date'].dt.quarter       # 분기 (1~4)

# ── 날짜 필터링 ────────────────────────────────────────────
# between: 시작일, 종료일 모두 포함 (이상 이하)
df_h1 = df[df['date'].between('2025-01-01', '2025-06-30')]

# 특정 연도/월만 추출
df_2026 = df[df['date'].dt.year == 2026]
df_jan  = df[df['date'].dt.month == 1]

# ── 날짜 차이 계산 ──────────────────────────────────────────
# 예: 날짜 차이를 일(day) 단위로 계산
df['days_diff'] = (df['order_date'] - df['join_date']).dt.days

# ── 최근/최초 날짜 ──────────────────────────────────────────
latest_date   = df['date'].max()   # 가장 최근 날짜
earliest_date = df['date'].min()   # 가장 오래된 날짜

## 6. 기술 통계량 및 수치 연산

In [ ]:
# ── 대표값 ────────────────────────────────────────────────
df['col'].mean()          # 평균
df['col'].median()        # 중앙값
df['col'].mode()[0]       # 최빈값

# ── 산포도 ────────────────────────────────────────────────
df['col'].var()           # 분산 (기본: ddof=1, 표본분산)
df['col'].std()           # 표준편차 (기본: ddof=1, 표본표준편차)
df['col'].max() - df['col'].min()  # 범위 (Range)

# ── 분포 형태 ──────────────────────────────────────────────
df['col'].skew()          # 왜도: 양수=오른쪽 꼬리, 음수=왼쪽 꼬리
df['col'].kurt()          # 첨도: pandas는 초과첨도(excess kurtosis) 반환
                          # (정규분포 기준 0 / scipy는 fisher=True가 기본)

# ── 분위수 ────────────────────────────────────────────────
df['col'].quantile(0.25)  # 1사분위수 (Q1)
df['col'].quantile(0.75)  # 3사분위수 (Q3)
df['col'].quantile(0.9)   # 상위 10% 경계값

# ── 기타 ─────────────────────────────────────────────────
df['col'].sum()                    # 합계
df['col_abs'] = df['col'].abs()    # 절대값 (반드시 변수에 저장해야 반영됨)
df['col'].count()                  # 결측치 제외한 데이터 개수
len(df['col'])                     # 전체 행 수 (결측치 포함)
df['col'].nunique()                # 고유값 개수

## 7. 조건 필터링 및 정렬

In [ ]:
# ── 단일 / 다중 조건 필터링 ──────────────────────────────────
cond_age    = (df['age'] >= 30)
cond_gender = (df['gender'] == 'M')

df_filtered = df[cond_age & cond_gender]    # AND 조건
df_or       = df[cond_age | cond_gender]    # OR 조건
df_not      = df[~cond_age]                 # NOT 조건

# ── isin으로 여러 값 필터링 ─────────────────────────────────
df_city = df[df['city'].isin(['서울', '부산', '대구'])]

# ── 비율 계산 ─────────────────────────────────────────────
# 예: 특정 값이 전체에서 차지하는 비율
ratio = (df['col'] == 6).sum() / len(df)

# ── 정렬 (sort_values) ───────────────────────────────────
# ascending=True: 오름차순(기본값), ascending=False: 내림차순
df_asc  = df.sort_values('income')                       # 오름차순
df_desc = df.sort_values('income', ascending=False)      # 내림차순

# 여러 컬럼 기준 정렬 (city 오름차순 → income 내림차순)
df_multi = df.sort_values(['city', 'income'], ascending=[True, False])

# ── 상위/하위 N개 추출 ──────────────────────────────────────
# 예: 매출 상위 10% 경계값 이상인 데이터의 평균
top_10pct_val = df['income'].quantile(0.9)
top_mean      = df[df['income'] >= top_10pct_val]['income'].mean()

# 예: 매출 상위 10명의 합계 (정수 출력)
top10_sum = df.sort_values('income', ascending=False)['income'].head(10).sum()
print(int(top10_sum))

# 예: iloc로 첫 10행의 평균 (반올림하여 정수 출력)
mean_10 = df.iloc[:10]['col'].mean()
print(round(mean_10))

## 8. 행/열 조작 및 파생변수 생성

> 기존 데이터를 기반으로 새로운 변수를 만들거나 불필요한 변수를 제거

In [ ]:
# ── 인덱싱 ───────────────────────────────────────────────
# loc: 라벨(이름) 기반, 끝 인덱스 포함
df.loc[3, 'col']                # 라벨 인덱스 3행, 'col' 열
df.loc[0:3, ['col1', 'col2']]   # 0~3행(3행 포함), 두 컬럼 추출
# iloc: 위치(숫자) 기반, 끝 인덱스 미포함  ← 파이썬 슬라이싱과 동일
df.iloc[0:3, 0:2]               # 위치 기반 0~2행(3행 미포함), 0~1열(2열 미포함)

# ── 열 제거 (drop) ────────────────────────────────────────
df = df.drop(['col1', 'col2'], axis=1)   # 열 삭제
df = df.drop(0, axis=0)                  # 0번 인덱스 행 삭제

# ── 파생변수 생성 ──────────────────────────────────────────
df['new_col'] = df['col'] + 10           # 사칙연산

# 조건 분기 (np.where)
# np.where(조건, 참일 때 값, 거짓일 때 값)
df['pass'] = np.where(df['score'] >= 60, 1, 0)

# 중첩 조건 (A이면 'high', B이면 'mid', 나머지 'low')
df['grade'] = np.where(df['score'] >= 80, 'high',
              np.where(df['score'] >= 60, 'mid', 'low'))

# ── 데이터프레임 복사 ────────────────────────────────────────
df_copy = df.copy()   # 원본 영향 없음

## 9. 그룹별 집계 연산과 데이터 결합

> 9-1. grouby()

In [ ]:
# ── groupby: 그룹별 요약 통계 ───────────────────────────────
# 결과는 그룹 수만큼의 행을 가진 요약 DataFrame
grouped_mean = df.groupby('city')['income'].mean()
# 예시 결과:
# city
# 서울    350.2   ← 서울 전체의 평균 한 줄
# 부산    290.1
# 대구    310.5

# 다중 그룹, 다중 집계 (agg)
grouped_agg = df.groupby(['city', 'gender']).agg(
    income_mean   = ('income', 'mean'),
    income_median = ('income', 'median'),
    count         = ('income', 'count'))

# ── transform: 그룹 통계를 원본 행 수 그대로 반환 ──────────────
# 결과는 원본 DataFrame과 동일한 행 수 → 새 컬럼으로 바로 붙이기 가능
df['city_mean'] = df.groupby('city')['income'].transform('mean')
# 예시 결과 (원본 행에 그대로 매핑):
# city   income  city_mean
# 서울    400     350.2     ← 서울 평균이 각 행에 채워짐
# 서울    300     350.2
# 부산    290     290.1
# 부산    290     290.1

# → groupby: 그룹 요약표가 필요할 때
# → transform: 원본 행에 그룹 통계를 추가 컬럼으로 붙이고 싶을 때
#  (예: 평균 대비 편차 계산, 조건 필터링 등)
df['diff_from_mean'] = df['income'] - df['city_mean']

# 그룹 수 확인
print(df.groupby('city').ngroups)

> 9-2. pd.concat() & pd.merge()

In [ ]:
# ── 데이터프레임 결합 ───────────────────────────────────────
# (1) 행 방향(위↓아래) 결합
concat_rows = pd.concat([df1, df2], axis=0).reset_index(drop=True)

# (2) 열 방향(좌→우) 결합
concat_cols = pd.concat([df1, df2], axis=1)

# (3) 키 기준 병합 (SQL JOIN과 동일)
# how: 'inner'(교집합), 'left', 'right', 'outer'(합집합)
merged_df = pd.merge(df1, df2, on='id', how='inner')
merged_df = pd.merge(df1, df2, left_on='user_id', right_on='id', how='left')

## 10. 중복 제거, 누적합, 피벗, 순위

In [ ]:
# ── 중복 제거 (drop_duplicates) ──────────────────────────
df['is_dup'] = df.duplicated(subset=['col'])  # 특정 칼럼의 중복 여부 확인
unique_count = df.drop_duplicates().shape[0]    # 중복 제거 후 행 수
drop_df = df.drop_duplicates(subset=['col'])  # 컬럼 기준 중복 제거
drop_df = df.drop_duplicates(subset=['col'], keep='last') # 마지막 값 유지
drop_df = df.drop_duplicates(subset=['col'], keep='first')# 기본값: 첫 값 유지
drop_df = df.drop_duplicates(subset=['col'], keep= False) # 중복된 데이터 모두 제거하고 단 한 번만 등장한 고유한 데이터만 남김

# ── 누적합 (cumsum) ──────────────────────────────────────
df['cumulative_income'] = df['income'].cumsum()

# ── 피벗 테이블 ───────────────────────────────────────────
# index=행, columns=열, values=집계대상, aggfunc=집계함수
pivot_df = df.pivot_table(
    index='city',
    columns='gender',
    values='income',
    aggfunc='mean',
    fill_value=0
)

# ── 순위 구하기 (rank) ────────────────────────────────────
# ascending=True: 작은 값이 1등 (기본값) / ascending=False: 큰 값이 1등
# method 옵션:
#     'average': 동점자 순위 평균 (기본값)
#     'min'    : 동점자에게 최소값 부여 (예: 2등과 3등 중 **2등**) (1124 방식)
#     'max'    : 동점자에게 최대값 부여 (1244 방식)
#     'dense'  : 동점자에게 동일 순위, 다음 순위는 바로 이어짐 (1123 방식)
#     'first'  : 동점자 중 먼저 나온 데이터에 작은 등수 숫자 부여
df['rank'] = df['income'].rank(ascending=False, method='min')

# 특정 순위 이상인 행 필터링
df_top5 = df[df['rank'] <= 5]

## 11. 이상치(Outlier) 탐지 및 처리

> 11-1. IQR 방식

In [ ]:
# 1단계: 사분위수 및 IQR 계산
q1  = df['col'].quantile(0.25)
q3  = df['col'].quantile(0.75)
iqr = q3 - q1

# 2단계: 이상치 경계선 정의
lower_bound = q1 - 1.5 * iqr   # 하한선
upper_bound = q3 + 1.5 * iqr   # 상한선

# 3단계: 정상 데이터만 추출 (이상치 제거)
df_clean   = df[(df['col'] >= lower_bound) & (df['col'] <= upper_bound)]
clean_mean = df_clean['col'].mean()

# 이상치 데이터만 추출하여 개수 확인
outliers      = df[(df['col'] < lower_bound) | (df['col'] > upper_bound)]
outlier_count = len(outliers)

> 11-2. Z-score 방식 (정규분포 가정)

In [ ]:
from scipy import stats

# Z-score 절대값 계산
z_scores   = np.abs(stats.zscore(df['col'].dropna()))
# 일반적으로 |Z| > 3 이면 이상치로 판단
df_clean_z = df[z_scores <= 3]

> 11-3. 이상치 클리핑 (제거 대신 상/하한값으로 대체)

In [ ]:
# 방법 A: np.where 활용
df['col_clipped'] = np.where(df['col'] > upper_bound, upper_bound,
                    np.where(df['col'] < lower_bound, lower_bound, df['col']))

# 방법 B: clip 활용 (더 간결)
df['col_clipped'] = df['col'].clip(lower=lower_bound, upper=upper_bound)